In [ ]:

import os

# 1. Crear carpeta para credenciales de Kaggle
os.makedirs('/root/.kaggle', exist_ok=True)

# 2. Verificar si kaggle.json fue subido
if not os.path.exists('kaggle.json'):
    raise FileNotFoundError("Error: Debes subir el archivo kaggle.json antes de continuar.")

# 3. Copiar kaggle.json a la carpeta correspondiente
!cp kaggle.json /root/.kaggle/
!chmod 600 /root/.kaggle/kaggle.json

print("Autenticación lista.")

# 4. Descargar el dataset (MIAS Mammography)
print("Descargando dataset...")
!kaggle datasets download -d kmader/mias-mammography -p ./mammography --unzip

print("Descarga y extracción completada.")

# 5. Verificar archivos descargados
print("Contenido del dataset:")
!ls ./mammography

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from PIL import Image #Con PIL (ahora Pillow), permite cargar, manipular y guardar imágenes en diversos formatos.
im = Image.open("./mammography/all-mias/mdb001.pgm") #Se abre la imagen mdb001.pgm que se encuentra en la carpeta del dataset y se guarda la imagen en la variable im como un objeto de tipo PIL.Image.Image
im #Se imprime el contenido de la variable, en este caso se muestra la imagen.


In [ ]:
#--------------------- Preparar el entorno para el preprocesamiento -------------
import tensorflow as tf #Para crear y entrenar una red neuronal
import matplotlib.pyplot as plt #Para realización de graficos
import os # Para manejar rutas y archivos
from IPython.display import clear_output #Limpia la salida y es útil para mostrar actualizaciones
from tqdm.notebook import tqdm #Muestra barras de proceso elegantes
import cv2 #Para manejo de openCV
from sklearn.model_selection import train_test_split # De sklearn se usa para dividir datos en entrenamiento y prueba
import random #Para generar valores aleatorios
import pandas as pd
import numpy as np



In [ ]:
imgs_path = './mammography/all-mias/' #Se define una variable con la ruta donde se encuentran las imágenes de las mamografías

In [ ]:
print("Leyendo Dataframe")
info = pd.read_csv("./mammography/Info.txt", sep=" ") # Se carga el archivo de la información "Info", donde se indica que va separado por comas
info = info.drop('Unnamed: 7', axis=1) #Se elimina una columna extra sin nombre
info #Se muestra el contenido del dataframe


Preparación de etiquetas de clasificación

In [ ]:

#Diccionario de índices a ID
ids = {} #Se crea un diccionario que mapea el índice de cada fila (i) al nombre o ID de la imagen correspondiente
for i in range(len(info)):
    ids[i] = info.REFNUM[i] #Esto puede usarse para relacionar los índices del DataFrame con los archivos de imagen reales

In [ ]:
#Se realiza un etiquetado binario de normal y anormal
label = [] #Se crea una lista que contendrá 0 si es normal y 1 si es anormal
for i in range(len(info)):
    if info.CLASS[i] != 'NORM': #columna que indica si la imagen es normal ('NORM') o qué tipo de anomalía tiene (por ejemplo, 'CALC', 'MASS', etc.).
        label.append(1)
    else:
        label.append(0)

label = np.array(label) #Se convierte la lista a un arreglo, ya que es mas eficiente con los modelos de maching learning
print(f"Cant. Imagenes: {len(label)}, Normales = {len(label)-np.sum(label)}, No normales= {np.sum(label)}") #se muestra el resumen de las etiquetas: total de normales, total de anormales y total de normales

In [ ]:
# Importando nuestras imágenes
# Se definen las rutas de archivo de cada imagen de la lista
img_name = [] # Se crea una lista img_name que contiene las rutas completas de cada imagen

for i in range(len(label)):
	# Se crea la ruta completa concatenando la carpeta base, el identificador de la imagen y la extensión.
        img_name.append(imgs_path + info.REFNUM[i]+ '.pgm')


Filtrado y balanceo de clases

In [ ]:
# Se preparan variables para un proceso de reducción de datos normales
count = 0 # Contador de cuantas imágenes normales se han saltado
remove = True # Bandera que indica si todavía estamos "removiendo" imágenes normales
# Listas temporales donde se guardan etiquetas y rutas de imágenes seleccionadas
temp_label = []
temp_img_name = []
# Lógica de balanceo
for i, lbl in enumerate(label.tolist()): # Se recorren las etiquetas(label) junto con su índice.
    if lbl == 0 and remove == True: # Si la etiqueta es 0 (imagen normal) y todavía está activo remove == True, se incrementa un contador (count)
        count = count + 1
        if count >= 84: # Una vez que count es mayor o igual a 84, se desactiva remove y ya no se eliminan normales.
            remove = False
    else:
        temp_label.append(lbl)# Se van guardando solo las imágenes que queremos mantener en temp_img_name
        temp_img_name.append(img_name[i]) # Y sus etiquetas correspondientes en temp_label

En otras palabras: se descartan las primeras  84 imágenes normales, y luego se guardan todas las demás (normales y anormales). Esto es una técnica de downsampling para reducir la cantidad de normales y balancear un poco más las clases.

In [ ]:
# Reconstrucción de los arrays finales
# Se actualizan las variables finales (label y img_name)con las listas filtradas.
label = np.array(temp_label) # Arreglo NumPy con las nuevas etiquetas
img_name = temp_img_name

In [ ]:
img_name = np.array(img_name) # Arreglo NumPy con las ruras de imágenes seleccionadas.
img_name.shape # Muestra el número de imágenes que quedaron después del filtrado
# Por ejemplo, si había 322 imágenes al inicio, quizás ahora queden unas 238 (dependiendo del dataset original y de cuántas normales se filtraron).
#print(img_name)
print(f'Cantidad de direcciones de imagen {img_name.shape}') #El print da como mensaje "image addres amount 238"

In [ ]:
# Vamos a ver 25 imágenes random de nuestro dataset
def view_25_random_image(): # Se define una función que muestra 25 imágenes aleatorias (en una cuadricula de 5x5)
    fig = plt.figure(figsize = (15, 10))
    for i in range(25):
        rand = random.randint(0,len(label))
        ax = plt.subplot(5, 5, i+1) # Se crea la cuadricula de 5x5

        img = cv2.imread(img_name[rand], 0) # Se realiza la lectura en escala de grises
        img = cv2.resize(img, (256,256)) # Se redimensionan a 256x256 píxeles (normalización de tamaño).
        if label[rand] == 1:# Se coloca un titulo según su etiqueta (Normal o Not Normal)
            plt.title('Not Normal')
        else:
            plt.title('Normal')
        plt.tight_layout()
        plt.axis('off')
        plt.imshow(img)
    fig.savefig('random_25_image_fig.png') # Se guarda la figura como random_25_image_fig.png.

random_images = view_25_random_image() # Se le asigna a una variable la salida de la función para mostrar las 25 imágenes aleatorias

# Preprocesamiento extra

---



In [ ]:
#Preprocesamiento para mejores resultados
import os
from tqdm.notebook import tqdm

output_dir = "./preprocessed_mias/"
os.makedirs(output_dir, exist_ok=True)

IMG_SIZE = 256

def preprocess_image(path):
    """Aplica preprocesamiento completo a una mamografía."""
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        return None

    # 1. Eliminación de fondo
    _, mask = cv2.threshold(img, 10, 255, cv2.THRESH_BINARY)
    img = cv2.bitwise_and(img, mask)

    # 2. Eliminación del músculo pectoral
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if contours:
        c = max(contours, key=cv2.contourArea)
        mask_clean = np.zeros_like(img)
        cv2.drawContours(mask_clean, [c], -1, 255, -1)
        img = cv2.bitwise_and(img, mask_clean)

    # 2.5 Detección y eliminación de etiquetas
    # Paso 1: Umbral adaptativo (resalta zonas brillantes)
    th = cv2.adaptiveThreshold(img, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                               cv2.THRESH_BINARY, 51, -10)
    th = cv2.bitwise_not(th)

    # Paso 2: Cerrar huecos pequeños para completar los bordes
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (7, 7))
    closed = cv2.morphologyEx(th, cv2.MORPH_CLOSE, kernel, iterations=2)

    # Paso 3: Buscar contornos
    contours, _ = cv2.findContours(closed, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    mask_shapes = np.ones_like(img, dtype=np.uint8) * 255
    img_area = img.shape[0] * img.shape[1]
    h, w = img.shape

    for cnt in contours:
        x, y, cw, ch = cv2.boundingRect(cnt)
        area = cv2.contourArea(cnt)
        aspect_ratio = cw / float(ch)

        # Condiciones: forma rectangular + tamaño medio + posición superior
        if 0.8 < aspect_ratio < 6.0 and 0.002 * img_area < area < 0.2 * img_area:
            if y < 0.35 * h and (x < 0.3 * w or x + cw > 0.7 * w):
                cv2.rectangle(mask_shapes, (x, y), (x + cw, y + ch), 0, -1)

    # Aplicar máscara sobre la imagen original
    img = cv2.bitwise_and(img, mask_shapes)


    # 3. Reducción de ruido y realce
    img = cv2.medianBlur(img, 5)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    img = clahe.apply(img)

    # 4. Máscara de enfoque
    blur = cv2.GaussianBlur(img, (9, 9), 10.0)
    img = cv2.addWeighted(img, 1.5, blur, -0.5, 0)

    # 5. Redimensionar y normalizar
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    img = img.astype(np.float32) / 255.0
    img = img.reshape(IMG_SIZE, IMG_SIZE, 1)  # Mantener canal único

    return img

In [ ]:
# Procesar todas las imágenes seleccionadas
processed_images = []
processed_labels = []

print("Procesando imágenes...")

for path, lbl in tqdm(zip(img_name, label), total=len(img_name)):
    img = preprocess_image(path)
    if img is not None:
        processed_images.append(img)
        processed_labels.append(lbl)

processed_images = np.array(processed_images)
processed_labels = np.array(processed_labels)

print(f"Imágenes preprocesadas: {processed_images.shape}")
print(f"Normales: {len(processed_labels) - np.sum(processed_labels)}, Anormales: {np.sum(processed_labels)}")

# Guardar imagenes para uso posterior
for i, img in enumerate(processed_images):
    out_path = os.path.join(output_dir, f"prep_{i:04d}.png")
    cv2.imwrite(out_path, (img * 255).astype(np.uint8))

print("Preprocesamiento completo. Carpeta generada: /preprocessed_mias/")

In [ ]:
# Visualizar imagenes procesadas
plt.figure(figsize=(15, 10))
for i in range(25):
    idx = random.randint(0, len(processed_images)-1)
    plt.subplot(5, 5, i+1)
    plt.imshow(processed_images[idx], cmap='gray')
    plt.title('Not Normal' if processed_labels[idx]==1 else 'Normal')
    plt.axis('off')
plt.suptitle("Ejemplos de mamografías preprocesadas", fontsize=16)
plt.show()

Aumento de Datos

Dado que se tienen aproximadamente 300 imagene, vamos a realizar un aumento de los datos por rotación.

In [ ]:
#from tensorflow.keras.preprocessing.image import ImageDataGenerator
import numpy as np


IMG_SIZE = 128
num_rotations = 45  # rotaciones cada 15°
zoom_range = 0.1
shift_range = 0.1

aug_dir = "./augmented_images/"
os.makedirs(aug_dir, exist_ok=True)

img_path_aug = []
labels_aug = []

print("Generando imágenes aumentadas...")

for i in range(len(img_name)):
    img = cv2.imread(img_name[i], 0)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    rows, cols = img.shape

    for r in range(num_rotations):
        angle = r * (180 // num_rotations)
        M = cv2.getRotationMatrix2D((cols / 2, rows / 2), angle, 1)
        img_rotated = cv2.warpAffine(img, M, (IMG_SIZE, IMG_SIZE))

        # Guardar imagen rotada
        filename = f"{aug_dir}img_{i}_rot{angle}.png"
        cv2.imwrite(filename, img_rotated)
        img_path_aug.append(filename)
        labels_aug.append(label[i])

        # Flip horizontal
        img_flip_h = cv2.flip(img_rotated, 1)
        filename_h = f"{aug_dir}img_{i}_rot{angle}_fh.png"
        cv2.imwrite(filename_h, img_flip_h)
        img_path_aug.append(filename_h)
        labels_aug.append(label[i])

        # Flip vertical
        img_flip_v = cv2.flip(img_rotated, 0)
        filename_v = f"{aug_dir}img_{i}_rot{angle}_fv.png"
        cv2.imwrite(filename_v, img_flip_v)
        img_path_aug.append(filename_v)
        labels_aug.append(label[i])

        # Zoom centrado
        zoom_factor = 1 + np.random.uniform(-zoom_range, zoom_range)
        new_size = int(IMG_SIZE * zoom_factor)
        img_zoom = cv2.resize(img_rotated, (new_size, new_size))

        # Recortar o rellenar al tamaño original
        if zoom_factor >= 1:
            start = (new_size - IMG_SIZE)//2
            img_zoom_crop = img_zoom[start:start+IMG_SIZE, start:start+IMG_SIZE]
        else:
            img_zoom_crop = np.zeros((IMG_SIZE, IMG_SIZE), dtype=np.uint8)
            start = (IMG_SIZE - new_size)//2
            img_zoom_crop[start:start+new_size, start:start+new_size] = img_zoom

        filename_z = f"{aug_dir}img_{i}_rot{angle}_zoom.png"
        cv2.imwrite(filename_z, img_zoom_crop)
        img_path_aug.append(filename_z)
        labels_aug.append(label[i])

        # Traslación simple
        tx = int(shift_range * IMG_SIZE * np.random.uniform(-1,1))
        ty = int(shift_range * IMG_SIZE * np.random.uniform(-1,1))
        M_shift = np.float32([[1,0,tx],[0,1,ty]])
        img_shift = cv2.warpAffine(img_rotated, M_shift, (IMG_SIZE, IMG_SIZE))
        filename_s = f"{aug_dir}img_{i}_rot{angle}_shift.png"
        cv2.imwrite(filename_s, img_shift)
        img_path_aug.append(filename_s)
        labels_aug.append(label[i])

print("Generación y guardado de imágenes aumentadas completada.")
print(f"Total imágenes aumentadas: {len(img_path_aug)}")

In [ ]:
# Convertir las listas en arrays
img_path_aug = np.array(img_path_aug)
labels_aug = np.array(labels_aug)

print("Forma de las rutas:", img_path_aug.shape)
print("Forma de las etiquetas:", labels_aug.shape)


Generación de dataframe con etiquetas

In [ ]:

# Crear DataFrame con rutas y etiquetas
df_aug = pd.DataFrame({
    "filename": img_path_aug,
    "label": labels_aug
})

# Guardar CSV en la carpeta de imágenes aumentadas
csv_path = os.path.join(aug_dir, "etiquetas_aum.csv")
df_aug.to_csv(csv_path, index=False)

print(f"CSV de etiquetas guardado en: {csv_path}")
print(f"Total imágenes listadas en CSV: {len(df_aug)}")


In [ ]:
import pandas as pd

# Ruta del archivo CSV generado por tu código
csv_path = "./augmented_images/etiquetas_aum.csv"

# Leer el CSV
df = pd.read_csv(csv_path)

# Calcular el balance de clases
total = len(df)
normales = (df['label'] == 0).sum()
anormales = (df['label'] == 1).sum()

# Calcular proporciones
normales_pct = (normales / total) * 100
anormales_pct = (anormales / total) * 100

print(f"Total de imágenes: {total}")
print(f"Normales (0): {normales} ({normales_pct:.2f}%)")
print(f"Anormales (1): {anormales} ({anormales_pct:.2f}%)")

División de datos en entrenamiento, prueba y validación

In [ ]:
from sklearn.model_selection import train_test_split

# Cargar rutas y etiquetas
X = df_aug['filename'].values
y = df_aug['label'].values

# División en: 70% entrenamiento, 15% validacion
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42)

print("Tamaños de los conjuntos:")
print(f"Entrenamiento: {len(X_train)} imágenes")
print(f"Validación: {len(X_val)} imágenes")
print(f"Prueba: {len(X_test)} imágenes")


In [ ]:
def load_images(paths):
    data = []
    for path in paths:
        img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
        img = img.astype(np.float32) / 255.0
        data.append(img.reshape(IMG_SIZE, IMG_SIZE, 1))
    return np.array(data)

x_train = load_images(X_train)
x_val   = load_images(X_val)
x_test  = load_images(X_test)

In [ ]:
print("Listo para entrenamiento:")
print(f"x_train: {x_train.shape}, y_train: {y_train.shape}")
print(f"x_val: {x_val.shape}, y_val: {y_val.shape}")
print(f"x_test: {x_test.shape}, y_test: {y_test.shape}")

print("Tipo:", x_train.dtype)
print("Shape:", x_train.shape)
print("Ejemplo de valor medio:", x_train.mean())


stratify=y asegura que la proporción de clases
(0/1) se mantenga en todos los subconjuntos.

El parámetro random_state=42 solo garantiza reproducibilidad.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers, activations
from keras.callbacks import TensorBoard
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from time import time

# Definición del modelo CNN

model = models.Sequential([
    # Bloque 1
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(IMG_SIZE, IMG_SIZE, 1)),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2,2),

    # Bloque 2
    layers.Conv2D(64, (3,3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2,2),

    # Bloque 3
    layers.Conv2D(64, (3,3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2,2),
    layers.Dropout(0.25),

    # Clasificación
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.25),
    layers.BatchNormalization(),
    layers.Dense(1, activation='sigmoid')
])

# Compilación
optimizer = tf.keras.optimizers.Adam(learning_rate=1e-4)
model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])


In [ ]:
# Resumen
model.summary()

Trining

In [ ]:
# Callbacks (para entrenamiento)
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# Early stopping: se detiene si no mejora la validación
early_stop = EarlyStopping(
    monitor='val_loss',
    mode='min',
    patience=5,
    restore_best_weights=True,
    verbose=1
)

# Guardar el mejor modelo
model_check_point = ModelCheckpoint(
    filepath='./mejor_modelo.h5',
    monitor='val_loss',
    verbose=1,
    save_best_only=True,
    mode='min'
)

Entrenamiento de la red neuronal

In [ ]:
train = True
if train:
    history = model.fit(
        x_train, y_train,
        validation_data=(x_val, y_val),
        epochs=100,
        batch_size=128,
        callbacks=[early_stop, model_check_point]
    )
else:
    model = tf.keras.models.load_model('./mejor_modelo.h5')


Observación de resultados

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12,5))

# Exactitud
plt.subplot(1,2,1)
plt.plot(history.history['accuracy'], label='Entrenamiento')
plt.plot(history.history['val_accuracy'], label='Validación')
plt.title('Precisión durante el entrenamiento')
plt.xlabel('Épocas')
plt.ylabel('Precisión')
plt.legend()

# Pérdida
plt.subplot(1,2,2)
plt.plot(history.history['loss'], label='Entrenamiento')
plt.plot(history.history['val_loss'], label='Validación')
plt.title('Pérdida durante el entrenamiento')
plt.xlabel('Épocas')
plt.ylabel('Pérdida')
plt.legend()

plt.show()


Evaluación y predicción

In [ ]:
from random import randint
from IPython.display import display

print("\nEvaluación global del modelo")

# Evaluar desempeño general en el conjunto de prueba
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=2)
print(f"\nPrecisión en conjunto de prueba: {test_acc:.4f}")
print(f"Pérdida en conjunto de prueba: {test_loss:.4f}")

In [ ]:
# Predicción en una sola imagen
# Se elige una imagen alatoria del conjunto de prueba
test_num = randint(0, len(x_test) -1)
sample = x_test[test_num]

# Realizar predicción
result = model.predict(np.expand_dims(sample, axis=0), verbose=0)

# Procesar predicción
pred_label = 1 if result[0][0] >= 0.5 else 0
real_label = int(y_test[test_num])

Visualizar resultado

In [ ]:
plt.figure(figsize=(4,4))
plt.imshow(sample.squeeze(), cmap='gray')
plt.axis('off')

color = 'green' if pred_label == real_label else 'red'
plt.title(f"Pred: {pred_label} (prob={result[0][0]:.2f}) | Real: {real_label}", color=color)
plt.show()

# Mostrar resumen de la predicción
print("Resultado obtenido")
print(f"Índice de imagen: {test_num}")
print(f"Probabilidad estimada: {result[0][0]:.4f}")
print(f"Etiqueta predicha: {pred_label}")
print(f"Etiqueta real: {real_label}")
print(f"Coincidencia: {'Correcto' if pred_label == real_label else 'Incorrecto'}")

Detectar y mostrar errores de clasificación

In [ ]:
# 1️. Predecir TODO el conjunto de prueba
y_pred_probs = model.predict(x_test, verbose=0)
y_pred = (y_pred_probs >= 0.5).astype(int)  # convertir probabilidades a 0 y 1

# 2️. Índices donde el modelo FALLÓ
errores = np.where(y_pred.reshape(-1) != y_test.reshape(-1))[0]

print(f"\nTotal de errores encontrados: {len(errores)} / {len(x_test)}")
print(f"Porcentaje de error: {len(errores) / len(x_test):.4f}")

# 3️. Mostrar algunas imágenes donde falló (hasta 9)
num_to_show = min(len(errores), 9)

if num_to_show == 0:
    print("\nEl modelo no tiene errores visibles en este conjunto de prueba.")
else:
    print("\nMostrando ejemplos de errores:")

    plt.figure(figsize=(12, 12))

    for i, idx in enumerate(errores[:num_to_show]):
        plt.subplot(3, 3, i + 1)
        plt.imshow(x_test[idx].squeeze(), cmap='gray')
        plt.title(f"Pred: {y_pred[idx][0]} | Real: {y_test[idx]}")
        plt.axis('off')

    plt.tight_layout()
    plt.show()


Guardar modelo :D

In [ ]:
save_model = True
if save_model:
  model.save("modelo_final_entrenado.h5")
  print("\n Modelo final guardado como 'modelo_final_entenado.h5 :D'")

Hemos llegado al final :0